In [9]:
import pandas as pd
from sklearn.linear_model import Ridge

train_df = pd.read_csv('../data/train.csv')
train_df.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,WeatherBucket,year,round_number,location
0,0 days 01:06:03.614000,VER,33,101.750,2.0,1.0,NaN,NaN,0 days 00:00:27.412000,0 days 00:00:40.860000,...,1,2.0,False,NaN,False,True,1,2021,Austin,Austin
1,0 days 01:07:44.885000,VER,33,101.271,3.0,1.0,NaN,NaN,0 days 00:00:27.350000,0 days 00:00:40.396000,...,1,2.0,False,NaN,False,True,1,2021,Austin,Austin
2,0 days 01:09:26.010000,VER,33,101.125,4.0,1.0,NaN,NaN,0 days 00:00:27.164000,0 days 00:00:40.479000,...,1,2.0,True,TRACK LIMITS AT TURN 19,False,True,1,2021,Austin,Austin
3,0 days 01:11:07.492000,VER,33,101.482,5.0,1.0,NaN,NaN,0 days 00:00:27.155000,0 days 00:00:40.724000,...,1,2.0,False,NaN,False,True,1,2021,Austin,Austin
4,0 days 01:12:49.322000,VER,33,101.830,6.0,1.0,NaN,NaN,0 days 00:00:27.445000,0 days 00:00:40.746000,...,1,2.0,False,NaN,False,True,1,2021,Austin,Austin


In [10]:
train_df.columns.tolist()


['Time',
 'Driver',
 'DriverNumber',
 'LapTime',
 'LapNumber',
 'Stint',
 'PitOutTime',
 'PitInTime',
 'Sector1Time',
 'Sector2Time',
 'Sector3Time',
 'Sector1SessionTime',
 'Sector2SessionTime',
 'Sector3SessionTime',
 'SpeedI1',
 'SpeedI2',
 'SpeedFL',
 'SpeedST',
 'IsPersonalBest',
 'Compound',
 'TyreLife',
 'FreshTyre',
 'Team',
 'LapStartTime',
 'LapStartDate',
 'TrackStatus',
 'Position',
 'Deleted',
 'DeletedReason',
 'FastF1Generated',
 'IsAccurate',
 'WeatherBucket',
 'year',
 'round_number',
 'location']

In [11]:
train_df['TrackStatus'].unique()


array([ 1, 12, 21,  2])

In [12]:
train_df['Driver'].value_counts()

Driver
VER    17274
HAM    16987
NOR    16966
RUS    16861
ALO    16717
SAI    16395
TSU    16288
STR    16272
OCO    16121
LEC    16073
GAS    16047
BOT    13939
PER    13868
ALB    11729
RIC    10977
PIA     9900
ZHO     9648
MAG     9323
HUL     9283
VET     6965
LAT     6797
MSC     6362
SAR     5727
GIO     4067
RAI     3733
LAW     3525
MAZ     3463
BEA     2753
DEV     2751
COL     2325
HAD     2212
BOR     2190
ANT     2103
DOO      609
Name: count, dtype: int64

In [33]:
def Trackstatus_bucket(status):
    status = str(status)
    if '5' in status:
        return 'red'
    elif '2' in status:
        return 'yellow'
    else:
        return 'green'

train_df['TrackStatusBucket'] = train_df['TrackStatus'].apply(Trackstatus_bucket)
train_df['TrackStatusBucket'].value_counts()


TrackStatusBucket
green     313390
yellow     12860
Name: count, dtype: int64

In [34]:
Y = train_df['LapTime']
numeric_features = train_df[['TyreLife', 'WeatherBucket']]
print(numeric_features.head())
categorical_features = pd.get_dummies(train_df[['Compound', 'Driver', 'TrackStatusBucket', 'location']])
print(categorical_features.head())
X = pd.concat([numeric_features, categorical_features], axis = 1)
print(X.columns.tolist())
print(X.shape)
print(train_df.shape)
print(numeric_features.shape)
print(categorical_features.shape)
print(Y.shape)


   TyreLife  WeatherBucket
0       5.0              1
1       6.0              1
2       7.0              1
3       8.0              1
4       9.0              1
   Compound_HARD  Compound_INTERMEDIATE  Compound_MEDIUM  Compound_SOFT  \
0          False                  False             True          False   
1          False                  False             True          False   
2          False                  False             True          False   
3          False                  False             True          False   
4          False                  False             True          False   

   Compound_WET  Driver_ALB  Driver_ALO  Driver_ANT  Driver_BEA  Driver_BOR  \
0         False       False       False       False       False       False   
1         False       False       False       False       False       False   
2         False       False       False       False       False       False   
3         False       False       False       False       False       F

In [18]:
X = X.dropna()
Y = Y.loc[X.index]
model = Ridge(alpha=1.0)
model.fit(X, Y)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [35]:
dev_df = pd.read_csv('../data/dev.csv')
dev_df.head()
dev_df['TrackStatusBucket'] = dev_df['TrackStatus'].apply(Trackstatus_bucket)
dev_df['TrackStatusBucket'].value_counts()
numeric_features = dev_df[['TyreLife', 'WeatherBucket']]
print(numeric_features.head())
categorical_features = pd.get_dummies(dev_df[['Compound', 'Driver', 'TrackStatusBucket', 'location']])
print(categorical_features.head())
X_dev = pd.concat([numeric_features, categorical_features], axis = 1)
X_dev = X_dev.reindex(columns=X.columns, fill_value=0)
print(X_dev.shape)
print(X.columns.tolist())
print(X.shape)
print(train_df.shape)
print(numeric_features.shape)
print(categorical_features.shape)

   TyreLife  WeatherBucket
0       2.0              1
1       3.0              1
2       4.0              1
3       5.0              1
4       6.0              1
   Compound_HARD  Compound_INTERMEDIATE  Compound_MEDIUM  Compound_SOFT  \
0          False                  False             True          False   
1          False                  False             True          False   
2          False                  False             True          False   
3          False                  False             True          False   
4          False                  False             True          False   

   Compound_WET  Driver_ALB  Driver_ALO  Driver_ANT  Driver_BEA  Driver_BOR  \
0         False       False       False       False       False       False   
1         False       False       False       False       False       False   
2         False       False       False       False       False       False   
3         False       False       False       False       False       F

In [37]:
Y_dev = dev_df['LapTime']
X_dev = X_dev.dropna()
Y_dev = Y_dev.loc[X_dev.index]
print(X_dev.shape, Y_dev.shape)
predictions = model.predict(X_dev)

(109073, 71) (109073,)


In [38]:
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(Y_dev, predictions)
r2 = r2_score(Y_dev, predictions)
print(mse, r2)

5.5142319091963525 0.9478031313201933


In [45]:
numeric_features_simple = train_df[['TyreLife', 'WeatherBucket']]
categorical_features_simple = pd.get_dummies(train_df[['Compound', 'TrackStatusBucket', 'location']])
X_simple = pd.concat([numeric_features_simple, categorical_features_simple], axis=1)
Y_simple = train_df['LapTime']

In [46]:
X_simple = X_simple.dropna()
Y_simple = Y_simple.loc[X_simple.index]
model = Ridge(alpha=1.0)
model.fit(X_simple, Y_simple)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [47]:
dev_df = pd.read_csv('../data/dev.csv')
dev_df.head()
dev_df['TrackStatusBucket'] = dev_df['TrackStatus'].apply(Trackstatus_bucket)
dev_df['TrackStatusBucket'].value_counts()
numeric_features = dev_df[['TyreLife', 'WeatherBucket']]
print(numeric_features.head())
categorical_features = pd.get_dummies(dev_df[['Compound', 'TrackStatusBucket', 'location']])
print(categorical_features.head())
X_dev = pd.concat([numeric_features, categorical_features], axis = 1)
X_dev = X_dev.reindex(columns=X_simple.columns, fill_value=0)
print(X_dev.shape)
print(X.columns.tolist())
print(X.shape)
print(train_df.shape)
print(numeric_features.shape)
print(categorical_features.shape)

   TyreLife  WeatherBucket
0       2.0              1
1       3.0              1
2       4.0              1
3       5.0              1
4       6.0              1
   Compound_HARD  Compound_INTERMEDIATE  Compound_MEDIUM  Compound_SOFT  \
0          False                  False             True          False   
1          False                  False             True          False   
2          False                  False             True          False   
3          False                  False             True          False   
4          False                  False             True          False   

   Compound_WET  TrackStatusBucket_green  TrackStatusBucket_yellow  \
0         False                     True                     False   
1         False                     True                     False   
2         False                     True                     False   
3         False                     True                     False   
4         False                     T

In [48]:
Y_dev = dev_df['LapTime']
X_dev = X_dev.dropna()
Y_dev = Y_dev.loc[X_dev.index]
print(X_dev.shape, Y_dev.shape)
predictions = model.predict(X_dev)

(109073, 37) (109073,)


In [49]:
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(Y_dev, predictions)
r2 = r2_score(Y_dev, predictions)
print(mse, r2)

5.940409175453414 0.9437690030558308


In [50]:
tire_life_index = X.columns.get_loc('TyreLife')
print(model.coef_[tire_life_index])

-0.02148355733377171
